# The Catalyst Optimizer: Rule-Based and Cost-Based Optimizations

**Module**: 07_spark_sql_internals  
**Notebook**: 01_catalyst_optimizer.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Catalyst rewrites tree structures using rule-based transformations (Predicate Pushdown, Column Pruning, Constant Folding) and Cost-Based Optimization (CBO).
- **Why does this topic exist?**: Catalyst is the query optimizer that makes Spark SQL and DataFrames fast.
- **What problem does it solve?**: Understanding how Spark optimizes your queries behind the scenes.
- **Where is it used in real-world Data Engineering?**: Explaining query optimization in senior data engineering interviews.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Advanced |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | DataFrames, Spark SQL, logical plans |
| **Required depth** | Deep internals: can explain analysis, logical optimization, planning, and cost-based decisions |
| **Data Engineer** | Read optimized plans and recognize which rules changed the query. |
| **Senior Data Engineer** | Diagnose why an optimization did not apply and rewrite the query accordingly. |
| **Spark specialist** | Reason about Catalyst rules, statistics, cost models, and plan stability. |
| **Next topic** | `07_spark_sql_internals/02_logical_and_physical_plans.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Catalyst is the optimizer behind DataFrames and SQL. It resolves names, applies rule-based rewrites such as
predicate pushdown and constant folding, considers physical strategies, and emits a plan. Most "Spark ignores my
filter" problems are really "the filter is not in a form the rule can use" or "the statistics are missing".

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Analyzer** | Resolves names and types | Binds attributes to catalog objects |
| **Logical optimizer** | Rule-based rewrites | Predicate pushdown, column pruning, constant folding |
| **Physical planner** | Chooses operators | Join strategies, aggregation modes, exchange |
| **Cost model** | Estimates plan cost | Uses table/column statistics when available |
| **Rule** | Tree transformation | Matches a plan pattern and rewrites it |
| **Code generation** | Compiles expressions/operators | Whole-stage codegen for supported plans |

### How it works, step by step

1. The query is parsed into an unresolved logical plan.
2. The analyzer resolves tables, columns, functions, and types.
3. Logical optimization rules rewrite the tree.
4. The physical planner generates candidate physical plans.
5. The cost model selects a plan, often using statistics.
6. The selected plan is prepared for execution and code generation.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Catalyst Tree Transformation Pipeline
1. **Analysis**: Resolves column names against Metastore Catalog.
2. **Logical Optimization**: Applies rule-based pattern matching:
   - `PushDownPredicate`: Pushes filters closer to the data source.
   - `ColumnPruning`: Discards unneeded columns early.
   - `ConstantFolding`: Evaluates constant expressions at compile time (`1 + 1 -> 2`).
3. **Physical Planning**: Cost model selects algorithms (e.g. BroadcastHashJoin vs SortMergeJoin).
4. **Code Generation**: Tungsten compiles plan into Java bytecode.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Catalyst_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
# Example query illustrating Constant Folding and Filter Pushdown
q = orders.filter("quantity > (1 + 1)").select("order_id", "status")
print("=== Optimized Logical Plan showing Constant Folding ===")
q.explain(True)

---

## Experiment: Constant Folding Demonstration

- **1. What are we testing?**: Verify in the Optimized Logical Plan that (1 + 1) was evaluated to 2 before execution.
- **2. What do we expect?**: Optimized plan shows GreaterThan(quantity, 2).

In [ ]:
print(q._jdf.queryExecution().optimizedPlan().toString())

**3. What happened?**: Expression was reduced to constant 2.  
**4. Why did it happen?**: Catalyst rule ConstantFolding evaluated the literal arithmetic.  
**5. Practical Engineering Lesson**: Spark does not evaluate arithmetic expressions row-by-row at runtime if they can be evaluated once at compile time.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Filter that never reaches the source

- **Industry**: Retail
- **Scenario**: A query filters on a derived column and scans the full Parquet table.
- **Problem**: The predicate is expressed in a way that prevents pushdown, so the scan reads every row.
- **Why this concept is relevant**: Catalyst can push simple predicates to the file source; rewriting the filter to reference the source column enables pruning.
- **Implementation idea**: Compare `PushedFilters` and scanned bytes before and after the rewrite.
- **Expected result**: A scan that reads only the relevant row groups.
- **Alternative**: Materialize a filtered table.
- **Trade-offs**: A filtered table reduces repeated scans but adds storage and freshness cost.

### Use Case 2: Missing statistics, wrong join

- **Industry**: Telecom
- **Scenario**: The optimizer chooses a sort-merge join for a small dimension that would fit in memory.
- **Problem**: Without statistics, the cost model overestimates the dimension size and rejects broadcast.
- **Why this concept is relevant**: Collecting table/column statistics gives Catalyst the information needed to choose a broadcast join.
- **Implementation idea**: Run `ANALYZE TABLE ... COMPUTE STATISTICS` and compare the plan.
- **Expected result**: A broadcast join with lower shuffle volume.
- **Alternative**: Add an explicit broadcast hint.
- **Trade-offs**: A hint is immediate but bypasses the cost model and can become wrong as data grows.

---

## Business Value

- Queries that scan less data and shuffle less.
- Lower cost per analytical query.
- Fewer manual hints and fragile tuning.
- Better explanations for plan changes during reviews.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Read the analyzed and optimized plans, not only the final physical plan.
- Keep statistics fresh for tables used in joins.
- Write predicates on source columns when possible.
- Do not fight the optimizer with hints without measuring.
- Check whether a rule was blocked by a type cast, UDF, or non-deterministic expression.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Watch a predicate move

**Steps**

1. Write a query that filters a Parquet table and joins a small dimension.
2. Print the analyzed, optimized, and physical plans.
3. Locate the filter in each plan and check `PushedFilters` in the scan.

**What to observe**: The optimizer pushes the filter closer to the scan when the expression is eligible.

**Healthy result**: The scan reports the expected pushed filter and reads fewer bytes.

**Problematic result**: The filter stays above the scan because of a cast or a UDF.

**Engineering takeaway**: The plan shows whether an optimization actually applied.

---

## Trade-offs and Alternatives

### Advantages

- Automatic rewrites
- Works across DataFrame and SQL APIs
- Extensible rules

### Limitations

- Statistics can be stale
- Some expressions block pushdown
- Cost model is approximate

### When to use it

- Analytical queries
- Join planning
- Predicate pushdown

### When not to use it

- Assuming every filter is pushed
- Hints without evidence
- Ignoring statistics

### Alternatives inside Spark

- Analyzer
- Optimizer
- Physical planner
- AQE

### Alternatives outside Spark

- Warehouse optimizers
- Query accelerators
- Materialized views

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify Column Pruning
**Problem Statement**: Verify that unused columns in orders are pruned in the physical plan.  
**Input / Context**: orders.select('order_id')

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Combine Filters Rule
**Problem Statement**: Observe that two consecutive filter() calls are combined into one AND expression.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Cost-Based Optimizer Flag
**Problem Statement**: Print spark.sql.cbo.enabled setting.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The cast that blocked pushdown

**Scenario**: A filter on a string date column stops partition pruning after a cast is introduced.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Compare the filter expression with the partition column type.
2. Check whether the cast prevents the optimizer from matching the partition predicate.
3. Filter on the native type or adjust the storage type.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify Column Pruning):
```python
orders.select('order_id').explain()
```

#### Solution 2 (Combine Filters Rule):
```python
orders.filter('quantity > 1').filter('quantity < 10').explain()
```

#### Solution 3 (Cost-Based Optimizer Flag):
```python
print(spark.conf.get('spark.sql.cbo.enabled','false'))
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Catalyst Power**: Tree transformation framework.
- **Rules**: Predicate pushdown, column pruning, constant folding.

### Essential Technical Terms
- `Catalyst`: Extensible Scala query optimizer.
- `Rule[LogicalPlan]`: Pattern-matching optimization rule.

### Common Engineering Mistakes
- **Over-optimizing manually**: Catalyst already reorders filters and prunes columns.

### Interview Questions on This Topic
- **Q: What are the primary phases of the Catalyst optimizer?**
  - *A*: Analysis, Logical Optimization, Physical Planning, and Code Generation.


### More Interview Questions

- **Q: What does the Catalyst analyzer do?**
  - *A*: It resolves table and column references, binds functions and types, and produces an analyzed logical plan that later rules can optimize.
- **Q: Why do statistics matter for joins?**
  - *A*: The physical planner uses size estimates to choose broadcast versus sort-merge joins; missing or stale statistics can lead to a poor strategy.

### What to Study Next
Open 02_logical_and_physical_plans.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')